# 2/2 - Benchmark a checkpoint (Kaggle or Colab GPU)

Scores a trained checkpoint on both halves of the published grid - PU-Net (20
shapes) and PC-Net (10 shapes), each at 10K/50K points and 1/2/3% noise - and
prints the rows into the comparison table.

## Before running

Attach `pointdenoise-code`, `pointdenoise-data` and the dataset holding your
`best.pt`. Turn the GPU on. Roughly 1.5 h for both datasets.

Calibration runs first and asserts on CD: a harness that cannot reproduce a
published number produces results comparable to nothing, so there is no point
scoring a model with it.


In [ ]:
import glob, os, subprocess, sys

# Works on Kaggle and on Colab. Kaggle mounts datasets at /kaggle/input; Colab
# pulls them with kagglehub into /root/.cache/kagglehub. Search both rather
# than assuming, so the same notebook runs either place.
SEARCH_ROOTS = [
    "/kaggle/input",
    "/root/.cache/kagglehub",
    "/content",
    os.getcwd(),
]
SEARCH_ROOTS = [r for r in SEARCH_ROOTS if os.path.isdir(r)]

def show_tree(root, limit=25):
    n = 0
    for base, dirs, files in os.walk(root):
        depth = base.replace(root, "").count(os.sep)
        if depth > 3:
            continue
        print("  " * depth + os.path.basename(base) + "/")
        for f in files[:2]:
            print("  " * (depth + 1) + f)
        if len(files) > 2:
            print("  " * (depth + 1) + "... (%d files)" % len(files))
        n += 1
        if n > limit:
            print("  ...truncated")
            return

def find_containing(*markers):
    """First directory under any search root that holds one of `markers`."""
    for root in SEARCH_ROOTS:
        for base, dirs, files in os.walk(root):
            for m in markers:
                if m in dirs or m in files:
                    return base
    return None

CODE = find_containing("pointdenoise")
DATA = find_containing("examples", "PUNet", "PCNet")
CKPT = None
for root in SEARCH_ROOTS:
    for base, _, files in os.walk(root):
        for f in ("best.pt", "last.pt"):
            if f in files:
                CKPT = os.path.join(base, f)
                break
        if CKPT:
            break
    if CKPT:
        break

print("roots :", SEARCH_ROOTS)
print("code  :", CODE)
print("data  :", DATA)
print("ckpt  :", CKPT or "(none - training will start from scratch)")

if not CODE or not DATA:
    for root in SEARCH_ROOTS:
        print()
        print("=== " + root + " ===")
        show_tree(root)
    missing = "pointdenoise-code" if not CODE else "pointdenoise-data"
    raise SystemExit(
        "\nCould not find the " + missing + " dataset.\n"
        "On Kaggle: use 'Add Input' and attach it.\n"
        "On Colab: run kagglehub.dataset_download('<user>/" + missing + "') first.\n"
        "The trees above show what is actually present."
    )

sys.path.insert(0, CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "trimesh", "rtree"], check=False)

# Colab and Kaggle both give a T4, but Colab needs the runtime type set.
import torch
print()
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NO GPU - Colab: Runtime > Change runtime type > T4. Kaggle: Settings > Accelerator")

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content/out"
os.makedirs(OUT, exist_ok=True)
print("outputs ->", OUT)


In [ ]:
assert CKPT, "no checkpoint found - attach the dataset holding best.pt"
from pointdenoise.engine import load_model

model, ck = load_model(CKPT)
print(f"loaded {CKPT}")
print(f"  epoch {ck.get('epoch')}, best loss {ck.get('best'):.6f}")
print(f"  kwargs {ck.get('model_kwargs')}")


## Calibrate first

Scores the bilateral filter, whose numbers are in the published table. Each
metric is checked separately: run 1 passed on CD at 0.84x while P2M sat at
0.17x, so P2M measures something different from what the papers report and
must not be quoted until that is resolved.


In [ ]:
from pointdenoise.benchmark import calibrate, load_released_set

case = load_released_set(DATA, "PUNet", "sparse", 0.01)
r = calibrate(case)
for m in ("cd", "p2m"):
    print(f"  {m.upper():<4} ours {r['measured_'+m]:7.3f}  published {r['expected_'+m]:6.2f}"
          f"  ratio {r[m+'_ratio']:.2f}x  {'PASS' if r[m+'_ok'] else 'FAIL'}")
print(f"\n  quotable: {[m.upper() for m in r['comparable_metrics']]}")
print(f"  not quotable: {[m.upper() for m in r['uncalibrated_metrics']]}")
assert r["cd_ok"], "CD calibration failed - results would not be comparable"


In [ ]:
import json

import numpy as np
from pointdenoise.benchmark import NOISE_LEVELS, load_released_set, run_case
from pointdenoise.data import Shape
from pointdenoise.engine import denoise_cloud

def denoiser(points):
    shape = Shape(np.asarray(points), noisy=np.asarray(points))
    return denoise_cloud(model, shape, points_per_patch=256, batch_size=128, iters=1)

# Resumable: each (dataset, resolution, noise) cell is written to disk the
# moment it finishes, not batched up for one write at the end. A disconnect
# then costs at most the cell in progress - not every cell computed before it,
# which is what happened last time: 4 cells finished and printed, but nothing
# was on disk to resume from, so a fresh session would have recomputed them.
PARTIAL_PATH = OUT + "/bench_partial.json"

def load_partial():
    if os.path.exists(PARTIAL_PATH):
        with open(PARTIAL_PATH) as f:
            return json.load(f)
    return {}

def save_partial(partial):
    with open(PARTIAL_PATH, "w") as f:
        json.dump(partial, f, indent=2)

partial = load_partial()
results = {}

for dataset in ("PUNet", "PCNet"):
    ds_partial = partial.setdefault(dataset, {})
    scores, baseline = {}, {}

    for resolution in ("sparse", "dense"):
        for noise in NOISE_LEVELS:
            key = f"{resolution}|{noise}"

            if key in ds_partial:
                ours = ds_partial[key]["ours"]
                none = ds_partial[key]["noisy"]
                print(f"{dataset}/{resolution}/{noise:.0%}  (cached) ours CD {ours['cd']:7.4f}  "
                      f"noisy {none['cd']:7.4f}")
            else:
                try:
                    case = load_released_set(DATA, dataset, resolution, noise)
                except (FileNotFoundError, RuntimeError) as e:
                    print(f"skip {dataset}/{resolution}/{noise:.0%}: {e}")
                    continue
                _, ours = run_case(case, denoiser, with_p2m=True)
                _, none = run_case(case, lambda p: p, with_p2m=True)
                gain = (none["cd"] - ours["cd"]) / none["cd"] * 100
                print(f"{dataset}/{resolution}/{noise:.0%}  ours CD {ours['cd']:7.4f}  "
                      f"noisy {none['cd']:7.4f}  {gain:+5.1f}%", flush=True)

                ds_partial[key] = {"ours": ours, "noisy": none}
                save_partial(partial)  # write now - a disconnect costs only this cell

            scores[(resolution, noise)] = ours
            baseline[(resolution, noise)] = none

    if scores:
        results[dataset] = (scores, baseline)

print(f"\ncached at {PARTIAL_PATH} - re-running this cell after a disconnect skips finished cells")


In [ ]:
CAVEAT = (
    "CD is calibrated: this harness reproduces the published Bilateral CD to 0.84x\n"
    "on the same shapes, so the CD columns are comparable.\n\n"
    "P2M is NOT calibrated - 0.17x the published value for the same algorithm - so\n"
    "the P2M columns are shown because the layout calls for them, not as a claim.\n"
)

out = []
for dataset, (scores, baseline) in results.items():
    published = None if dataset == "PUNet" else {}
    table = paper_table(scores, our_name="Ours", dataset=dataset, published=published)
    print(table); print()
    out.append(table); out.append("")
    out.append(f"{dataset} noisy-input baseline (CD x1e-4)")
    for k, v in baseline.items():
        o = scores[k]["cd"]
        out.append(f"  {k[0]}/{k[1]:.0%}  ours {o:7.4f}  noisy {v['cd']:7.4f}  "
                   f"{(v['cd'] - o) / v['cd'] * 100:+5.1f}%")
    out.append("")

with open(OUT + "/benchmark.txt", "w") as f:
    f.write("\n".join(out) + "\n\n" + CAVEAT)
print("saved", OUT + "/benchmark.txt")
